In [1]:

# PGRFA COMPONENT ABLATION EXPERIMENT
# 3 methods x 5 seeds x 20 tasks
# Preserves the original 10-run experiment

import json
import subprocess
import zipfile
from pathlib import Path

ROOT = Path("/kaggle/working")
SOURCE_DIR = ROOT / "PGRFA_ABLATION_SOURCE"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

KERNEL = "siva1987d/notebook9f4545a7fd"
DRIVE_FILE_ID = "1dnXRMgOhlIcnie8lZKOkEnPghi9wtVaN"

# Retrieve the ORIGINAL notebook source.
notebooks = list(SOURCE_DIR.glob("*.ipynb"))

if not notebooks:
    command = [
        "kaggle", "kernels", "pull", KERNEL,
        "-p", str(SOURCE_DIR), "-m"
    ]
    try:
        result = subprocess.run(
            command, capture_output=True,
            text=True, timeout=180
        )
        print("Kaggle source retrieval:", result.returncode)
        if result.returncode:
            print(result.stderr[-1000:])
    except Exception as error:
        print("Kaggle CLI retrieval:", error)

    notebooks = list(SOURCE_DIR.glob("*.ipynb"))

# Alternative: use the shared Google Drive source.
if not notebooks:
    import requests

    url = (
        "https://drive.google.com/uc?export=download&id="
        + DRIVE_FILE_ID
    )
    response = requests.get(url, timeout=90)
    response.raise_for_status()

    candidate = SOURCE_DIR / "original_pgrfa.ipynb"
    candidate.write_bytes(response.content)

    try:
        json.loads(candidate.read_text(encoding="utf-8"))
        notebooks = [candidate]
    except Exception:
        raise RuntimeError(
            "Source retrieval failed. Attach the original "
            "notebook as a Kaggle input and retry."
        )

notebook = json.loads(
    notebooks[0].read_text(encoding="utf-8")
)

source = "\n\n".join(
    "".join(c["source"])
    if isinstance(c["source"], list)
    else c["source"]
    for c in notebook["cells"]
    if c["cell_type"] == "code"
)

def replace_once(old, new):
    global source
    count = source.count(old)
    if count != 1:
        raise RuntimeError(
            f"Source validation failed: expected one "
            f"occurrence, found {count}: {old}"
        )
    source = source.replace(old, new, 1)


# Three controlled component removals
replace_once(
    'METHODS = ["improved", "replay_only"]',
    'METHODS = ["no_feature", "no_prototype", "no_logit"]'
)

# Enable the combined loss formula for all ablation methods
replace_once(
    'if method == "improved":',
    'if method in METHODS:'
)

# Disable exactly one loss term in each experiment
replace_once(
    '+ FEATURE_WEIGHT * old["feature"]',
    '+ (0.0 if method == "no_feature" else FEATURE_WEIGHT)'
    ' * old["feature"]'
)

replace_once(
    '+ PROTOTYPE_WEIGHT * old["prototype"]',
    '+ (0.0 if method == "no_prototype" else PROTOTYPE_WEIGHT)'
    ' * old["prototype"]'
)

replace_once(
    '+ DISTILL_WEIGHT * old["distillation"]',
    '+ (0.0 if method == "no_logit" else DISTILL_WEIGHT)'
    ' * old["distillation"]'
)

# Never overwrite the original research results
source = source.replace(
    "FINAL_PGRFA_RESEARCH",
    "PGRFA_ABLATION_RESEARCH"
)
source = source.replace(
    "FINAL_PGRFA_RESULTS",
    "PGRFA_ABLATION_RESULTS"
)

# Adjust the report for 15 planned runs
source = source.replace("/10", "/15")
source = source.replace(
    "10 completed 20-task runs.",
    "15 completed 20-task runs."
)

replace_once(
    'run_folder("improved", SEEDS[0])',
    'run_folder("no_feature", SEEDS[0])'
)

# The existing full model is a reference, not a new run.
replace_once(
    "PUBLISHED_ACC = 73.8",
    "PUBLISHED_ACC = 82.008"
)
replace_once(
    "PUBLISHED_FORGETTING = 9.5",
    "PUBLISHED_FORGETTING = 7.115789473684212"
)

source = source.replace(
    "PUBLISHED COLONELADAPTAI REFERENCE:",
    "PREVIOUS FULL PGRFA REFERENCE:"
).replace(
    "Average accuracy: 73.80%",
    "Average accuracy: 82.008%"
).replace(
    "Average forgetting: 9.50%",
    "Average forgetting: 7.116%"
).replace(
    "Published reference (unmatched)",
    "Previous full PGRFA (5 seeds)"
).replace(
    "Matched replay-only baseline uses the same backbone.",
    "Each ablation removes exactly one added loss term."
).replace(
    "Improved: replay, prototype alignment,",
    "Ablations: one added loss term removed,"
).replace(
    "historical feature alignment and logit distillation.",
    "with all other method settings retained."
).replace(
    "Task Accuracy Matrix - Improved, Seed 42",
    "Task Accuracy Matrix - No Feature, Seed 42"
).replace(
    "Published comparison is descriptive, not matched.",
    "Full PGRFA reference comes from the earlier study."
).replace(
    "It cannot prove causal superiority.",
    "Ablation comparisons assess component contributions."
)

SCRIPT = SOURCE_DIR / "pgrfa_ablation.py"
SCRIPT.write_text(source, encoding="utf-8")

# Validate Python syntax before starting GPU training.
compile(source, str(SCRIPT), "exec")

print("\nSource validation: PASS")
print("Methods: no_feature, no_prototype, no_logit")
print("Seeds: 42, 43, 44, 45, 46")
print("Total: 15 runs, 300 task stages")
print("The original experiment will not be retrained.")

# Execute the original training framework with ablation changes.
exec(compile(source, str(SCRIPT), "exec"), globals())

# Prepare the final comparison against the saved full method.
FULL = {
    42: (81.89, 6.452631578947369),
    43: (81.80, 8.200000000000001),
    44: (82.44, 6.778947368421050),
    45: (81.89, 7.389473684210519),
    46: (82.02, 6.7578947368421005)
}

runs, _ = collect_runs()
comparisons = []

if not runs.empty:
    for _, r in runs.iterrows():
        if r["status"] != "completed":
            continue

        seed = int(r["seed"])
        full_acc, full_forget = FULL[seed]

        comparisons.append({
            "removed_component": r["method"],
            "seed": seed,
            "full_accuracy": full_acc,
            "ablation_accuracy": float(r["accuracy"]),
            "accuracy_contribution_pp":
                full_acc - float(r["accuracy"]),
            "full_forgetting": full_forget,
            "ablation_forgetting": float(r["forgetting"]),
            "forgetting_reduction_contribution_pp":
                float(r["forgetting"]) - full_forget
        })

import pandas as pd

OUT = ROOT / "PGRFA_ABLATION_RESEARCH"
comparison_df = pd.DataFrame(comparisons)

if not comparison_df.empty:
    comparison_df.to_csv(
        OUT / "ablation_comparisons.csv",
        index=False
    )

    print("\nABLATION RESULTS")
    print(
        comparison_df.groupby("removed_component")[
            [
                "accuracy_contribution_pp",
                "forgetting_reduction_contribution_pp"
            ]
        ].agg(["mean", "std"]).round(4)
    )

    print("\nPositive contribution values indicate that")
    print("the removed component benefited full PGRFA.")

# Full backup, including saved model checkpoints
FULL_ZIP = ROOT / "PGRFA_ABLATION_COMPLETE_BACKUP.zip"

with zipfile.ZipFile(
    FULL_ZIP, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as archive:

    for file in OUT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname="research/" + str(file.relative_to(OUT))
            )

    for file in SOURCE_DIR.iterdir():
        if file.is_file():
            archive.write(
                file,
                arcname="source/" + file.name
            )

from IPython.display import display, FileLink

print("\nExperiment status:", run_status)
print("Completed ablation runs:", len(comparison_df), "/15")
print("Full backup:", FULL_ZIP)
print("Backup size (MB):", round(
    FULL_ZIP.stat().st_size / 1024**2, 2
))

display(FileLink(str(FULL_ZIP)))


Kaggle source retrieval: 0

Source validation: PASS
Methods: no_feature, no_prototype, no_logit
Seeds: 42, 43, 44, 45, 46
Total: 15 runs, 300 task stages
The original experiment will not be retrained.
FIVE-SEED GPU CONTINUAL LEARNING EXPERIMENT
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Seeds: [42, 43, 44, 45, 46]
Methods: ['no_feature', 'no_prototype', 'no_logit']
[2026-11-06 21:34:04] Downloading official CIFAR-100 dataset


100%|██████████| 169M/169M [11:06<00:00, 253kB/s]


[2026-11-06 21:45:16] ORIGINAL CIFAR-100 VERIFIED: 50000 train, 10000 test
[2026-11-06 21:45:16] STARTING TASK 1 / 20 | METHOD no_feature | SEED 42
[2026-11-06 21:45:28] EPOCH 1 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 1.40562
[2026-11-06 21:45:32] EPOCH 3 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.95066
[2026-11-06 21:45:37] EPOCH 6 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.82293
[2026-11-06 21:45:42] EPOCH 9 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.74157
[2026-11-06 21:45:48] EPOCH 12 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.66182
[2026-11-06 21:45:54] EPOCH 15 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.577
[2026-11-06 21:45:59] EPOCH 18 / 18 TASK 1 METHOD no_feature SEED 42 LOSS 0.52667
[2026-11-06 21:46:01] TASK COMPLETED: no_feature seed 42 tasks 1 accuracy 82.0 forgetting 0.0 BWT 0.0 stored images 2000
[2026-11-06 21:46:01] STARTING TASK 2 / 20 | METHOD no_feature | SEED 42
[2026-11-06 21:46:04] EPOCH 1 / 18 TASK 2 METHOD no_feature SEED 42 LOSS 5.34877
[2026-11-

/kaggle/working/PGRFA_ABLATION_RESULTS.pdf

/kaggle/working/PGRFA_ABLATION_RESULTS.zip


All available results have been saved.
A 'complete' study requires 15 completed 20-task runs.

ABLATION RESULTS
                  accuracy_contribution_pp          \
                                      mean     std   
removed_component                                    
no_feature                           0.918  0.2951   
no_logit                             2.838  0.4216   
no_prototype                         0.318  0.4748   

                  forgetting_reduction_contribution_pp          
                                                  mean     std  
removed_component                                               
no_feature                                      0.8863  0.6313  
no_logit                                        2.6147  0.7018  
no_prototype                                    0.3600  0.5688  

Positive contribution values indicate that
the removed component benefited full PGRFA.

Experiment status: complete
Completed ablation runs: 15 /15
Full backup: /kaggle/wo

/kaggle/working/PGRFA_ABLATION_COMPLETE_BACKUP.zip